# Sifu demo: Design stage, risk router, pause and resume

Needs Ollama running with the models in `config.yaml` and `tripplanner-ios` at `workspace.product_repo_path`. Cell 2 runs the five seeded Design-stage tickets. Cells 3 to 5 show the router: a low-risk ticket is approved automatically and a ticket that names a protected file (`.github/`) pauses for you.

The databases in `data/` persist between runs, so tickets and open requests pile up. For a clean run, set `RESET = True` in the next cell, restart the kernel and run it before the others.

In [1]:
from pathlib import Path

RESET = False  # True deletes the demo databases and their checkpoints for a clean run

if RESET:
    # Restart the kernel first so no open connection still holds these files.
    for name in ("design.db", "sifu.db"):
        for path in Path("data").glob(f"{name}*"):  # the .db file plus its -wal and -shm
            path.unlink()
            print(f"removed {path}")

In [2]:
from sifu.demo import run_design_demo

rows = run_design_demo(db_path="data/design.db")
rows

ok clean         expected=spec      got=spec      status=spec         events=15
   /Users/lance.fu/Development/AgenticAI/sifu/data/artifacts/design/1/spec.md
   reviewer:scope       0 findings
   reviewer:ambiguity   0 findings
   reviewer:policy      1 findings
ok ambiguous     expected=questions got=questions status=needs_human  events=11
   /Users/lance.fu/Development/AgenticAI/sifu/data/artifacts/design/2/questions.md
   reviewer:scope       3 findings
   reviewer:policy      1 findings
   reviewer:ambiguity   3 findings
ok out_of_scope  expected=questions got=questions status=needs_human  events=11
   /Users/lance.fu/Development/AgenticAI/sifu/data/artifacts/design/3/questions.md
   reviewer:policy      3 findings
   reviewer:scope       1 findings
   reviewer:ambiguity   6 findings
ok contradiction expected=questions got=questions status=needs_human  events=11
   /Users/lance.fu/Development/AgenticAI/sifu/data/artifacts/design/4/questions.md
   reviewer:scope       2 findings
   

[{'key': 'clean',
  'ticket_id': 1,
  'expected': 'spec',
  'outcome': 'spec',
  'status': 'spec',
  'events': 15,
  'artifact': '/Users/lance.fu/Development/AgenticAI/sifu/data/artifacts/design/1/spec.md'},
 {'key': 'ambiguous',
  'ticket_id': 2,
  'expected': 'questions',
  'outcome': 'questions',
  'status': 'needs_human',
  'events': 11,
  'artifact': '/Users/lance.fu/Development/AgenticAI/sifu/data/artifacts/design/2/questions.md'},
 {'key': 'out_of_scope',
  'ticket_id': 3,
  'expected': 'questions',
  'outcome': 'questions',
  'status': 'needs_human',
  'events': 11,
  'artifact': '/Users/lance.fu/Development/AgenticAI/sifu/data/artifacts/design/3/questions.md'},
 {'key': 'contradiction',
  'ticket_id': 4,
  'expected': 'questions',
  'outcome': 'questions',
  'status': 'needs_human',
  'events': 11,
  'artifact': '/Users/lance.fu/Development/AgenticAI/sifu/data/artifacts/design/4/questions.md'},
 {'key': 'injection',
  'ticket_id': 5,
  'expected': 'questions',
  'outcome': 'qu

In [3]:
from sifu.demo import open_demo, run_router_demo

deps, project_id = open_demo(db_path="data/sifu.db")
ids = run_router_demo(deps, project_id)

ticket 1 (clean): continued
ticket 2 (protected_path): PAUSED for a human
decision 1 ticket 1 accept_spec: floor=0 final=1 rules=["assessor_disagreement"] -> auto_with_veto
decision 2 ticket 2 accept_spec: floor=2 final=2 rules=["touches_protected_path"] -> pending_human
OPEN request 1 (ticket 2): ["approve", "close"]


The second ticket is paused in the SQLite checkpoint. Look at the open request. If it asks you to approve, answer `approve` or `close`, or any other text as feedback. If it asks you to answer questions, answer with text (it goes back to the reviewers as a clarification) or `close`.

In [4]:
from sifu import db
from sifu.demo import answer_request

# Answer the request this run's protected ticket opened. A reused database file can hold
# older open requests too, so pick it by ticket rather than by a fixed id.
[request] = [r for r in db.open_requests(deps.conn) if r["ticket_id"] == ids["protected_path"]]
state = answer_request(deps, request["id"], "approve")

ticket 2: resumed, status=spec
decision 1 ticket 1 accept_spec: floor=0 final=1 rules=["assessor_disagreement"] -> auto_with_veto
decision 2 ticket 2 accept_spec: floor=2 final=2 rules=["touches_protected_path"] -> human_approved
